# Voltage clamp virtuale del modello MNTB – protocolli di Wang et al. 1998 (REF2)

**Obiettivo:** riprodurre *in silico* gli esperimenti di voltage clamp di Wang 1998 (Fig. 2 e Fig. 4)
sul modello `mntb_hh_kv3_neuron.nestml`, per validare le due correnti di potassio:

- **I_HT** = corrente ad alta soglia Kv3 (nel modello `I_Kv3`, gate n e p), sensibile a TEA 1 mM;
- **I_LT** = corrente a bassa soglia (gate l e r), sensibile a DTX 100 nM.

**Come funziona il voltage clamp qui.** In voltage clamp la tensione è imposta dallo sperimentatore,
quindi l'equazione di membrana sparisce e restano solo le equazioni dei gate. Con V costante a tratti,
ogni gate ha soluzione esatta

$$x(t) = x_\infty(V) + \big(x_0 - x_\infty(V)\big)\,e^{-t/\tau_x(V)},\qquad x_\infty=\frac{\alpha}{\alpha+\beta},\ \tau_x=\frac{1}{\alpha+\beta}$$

Non serve NEST e non c'è errore di integrazione. Le correnti si calcolano poi dai gate.
Il sodio è escluso perché Wang registra in TTX 0.5 µM. Il leak è escluso per default
(correnti "leak-subtracted"); si può reincludere con `INCLUDE_LEAK = True`.

**I parametri vengono letti direttamente dal file `.nestml`**: se cambiate il modello, il notebook
usa i valori nuovi senza ricopiarli a mano.

| Esperimento | Wang 1998 | Protocollo | Cosa si misura |
|---|---|---|---|
| VC1 | Fig. 2A-B | hold −80, prepulse −100 mV (30 ms), gradini −80…+60 mV (200 ms) ± TEA | corrente totale, I-V, blocco da TEA |
| VC2 | Fig. 2C-D | hold −40 mV (I_LT inattivata), stessi gradini ± TEA | I_HT isolata |
| VC3 | Fig. 2E-F | sottrazione VC1 − VC2 | I_LT "sperimentale" vs I_LT vera del modello |
| VC4 | Fig. 2G-H | hold −80 → −20 mV → −40 mV ± DTX; I-V ± DTX | I_LT isolata, blocco da DTX |
| VC5 | Fig. 4A-C | hold −40, gradini −80…+60 mV | tempo di salita 10–90% |
| VC6 | Fig. 4D-G | hold −40 → +40 mV → −20 mV ± TEA | costante di tempo della tail, inibizione da TEA |

I valori I-V sono presi a **195 ms** dall'inizio del gradino e normalizzati alla corrente di controllo
a +60 mV del protocollo VC1, come nella didascalia di Fig. 2.

In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ======================= impostazioni =======================
MODEL_FILE = "mntb_hh_kv3_neuron.nestml"   # oppure "mntb_hh_kv3_neuron_copy.nestml"
FIG_DIR = "figures"                         # cartella (relativa al notebook) dove salvare le figure
os.makedirs(FIG_DIR, exist_ok=True)

INCLUDE_LEAK = False   # True: aggiunge g_L*(V - E_L) alla corrente "registrata"
TEA_BLOCK = 0.75       # frazione di I_HT bloccata da TEA 1 mM (Wang: 75.0 ± 5.9 %, Fig. 4G; testo: 70-80 %)
DTX_BLOCK = 0.75       # frazione di I_LT bloccata da DTX 100 nM (Wang Fig. 2G: corrente residua 25.0 ± 2.1 %)
DT = 0.01              # ms, passo di campionamento delle tracce (la soluzione è esatta, serve solo per i grafici)
STEPS = np.arange(-80, 61, 20)   # tensioni dei gradini (mV), come in Wang
T_MEAS = 195.0                   # ms dall'inizio del gradino in cui si misura la I-V


# ======================= lettura parametri dal .nestml =======================
def read_nestml_parameters(path):
    """Legge i valori numerici del blocco `parameters:` di un file NESTML.

    Restituisce un dizionario {nome: valore}. Le unità sono quelle scritte nel file
    (nS, mV, pF, 1/ms, 1/mV): nel modello sono coerenti, quindi basta il numero.
    """
    params, inside = {}, False
    for line in open(path):
        code = line.split("#")[0].rstrip()
        if not code.strip():
            continue
        indent = len(code) - len(code.lstrip())
        if code.strip() == "parameters:":
            inside, block_indent = True, indent
            continue
        if inside:
            if indent <= block_indent:          # fine del blocco
                break
            m = re.match(r"\s*(\w+)\s+[\w/]+\s*=\s*([-+]?[0-9.]+(?:[eE][-+]?\d+)?)", code)
            if m:
                params[m.group(1)] = float(m.group(2))
    return params


P = read_nestml_parameters(MODEL_FILE)
GATES = ["l", "r", "n", "p"]                    # i gate del potassio (m, h del sodio non servono: TTX)
for g in GATES:
    for k in ("k_alpha_", "eta_alpha_", "k_beta_", "eta_beta_"):
        assert k + g in P, f"parametro {k + g} non trovato in {MODEL_FILE}"
print(f"Modello: {MODEL_FILE}")
print(f"g_Kv3 = {P['g_Kv3']} nS, g_LT = {P['g_LT']} nS, g_L = {P['g_L']} nS, "
      f"E_K = {P['E_K']} mV, E_L = {P['E_L']} mV, C_m = {P['C_m']} pF")

In [ ]:
# ======================= nucleo del voltage clamp =======================
def rates(g, V):
    """alpha, beta (1/ms) del gate g alla tensione V (mV): k*exp(eta*V)."""
    a = P["k_alpha_" + g] * np.exp(P["eta_alpha_" + g] * V)
    b = P["k_beta_" + g] * np.exp(P["eta_beta_" + g] * V)
    return a, b


def x_inf(g, V):
    a, b = rates(g, V)
    return a / (a + b)


def tau_x(g, V):
    a, b = rates(g, V)
    return 1.0 / (a + b)


def currents(gates, V, block=None):
    """Correnti in nA (positive = uscenti, convenzione delle figure di Wang).

    block: dizionario di frazioni bloccate, es. {"Kv3": 0.75} per TEA, {"LT": 0.75} per DTX.
    Per ora I_Kv3 è un'unica corrente: quando il modello avrà Kv3.1 e Kv3.3 separati
    basta aggiungere qui le due correnti (e i relativi blocchi).
    """
    block = block or {}
    s = gates
    I = {
        "Kv3": (1 - block.get("Kv3", 0)) * P["g_Kv3"] * s["n"]**3 * (0.8 + 0.2 * s["p"]) * (V - P["E_K"]) / 1000.0,
        "LT":  (1 - block.get("LT", 0)) * P["g_LT"] * s["l"] * s["r"] * (V - P["E_K"]) / 1000.0,
    }
    I["leak"] = P["g_L"] * (V - P["E_L"]) / 1000.0 if INCLUDE_LEAK else 0.0 * V
    I["total"] = I["Kv3"] + I["LT"] + I["leak"]
    return I


def clamp(protocol, block=None, dt=DT):
    """Simula un protocollo di voltage clamp.

    protocol: lista di (V in mV, durata in ms). Il primo tratto è il potenziale di holding:
              i gate partono dallo stato stazionario a quella tensione (holding mantenuto a lungo,
              come i 2 minuti a -40 mV di Wang Fig. 2C).
    Restituisce t (ms), V (mV), dizionario delle correnti (nA) e dei gate.
    """
    state = {g: x_inf(g, protocol[0][0]) for g in GATES}
    t_all, V_all, gates_all = [], [], {g: [] for g in GATES}
    t0 = 0.0
    for V, dur in protocol:
        n = max(int(round(dur / dt)), 1)
        tt = np.arange(n) * dt
        for g in GATES:
            xi, tau = x_inf(g, V), tau_x(g, V)
            gates_all[g].append(xi + (state[g] - xi) * np.exp(-tt / tau))
            state[g] = xi + (state[g] - xi) * np.exp(-n * dt / tau)   # stato alla fine del tratto
        t_all.append(t0 + tt)
        V_all.append(np.full(n, float(V)))
        t0 += n * dt
    t = np.concatenate(t_all)
    Vt = np.concatenate(V_all)
    gates = {g: np.concatenate(gates_all[g]) for g in GATES}
    return t, Vt, currents(gates, Vt, block), gates


def step_family(hold, steps=STEPS, prepulse=None, t_pre=10.0, t_step=200.0, t_post=20.0, block=None):
    """Famiglia di gradini: holding (t_pre) -> [prepulse] -> gradino (t_step) -> holding (t_post).

    Il tempo restituito è riferito all'inizio del gradino (t = 0).
    """
    out = {}
    for Vs in steps:
        prot = [(hold, t_pre)]
        if prepulse is not None:
            prot.append(prepulse)
        prot += [(Vs, t_step), (hold, t_post)]
        t, Vt, I, _ = clamp(prot, block)
        t_on = t_pre + (prepulse[1] if prepulse is not None else 0.0)
        out[Vs] = (t - t_on, Vt, I)
    return out


def value_at(fam, Vs, key="total", t_meas=T_MEAS):
    t, _, I = fam[Vs]
    return I[key][np.argmin(np.abs(t - t_meas))]


def iv(fam, key="total", steps=STEPS):
    return np.array([value_at(fam, Vs, key) for Vs in steps])


def plot_family(ax, fam, title, key="total", xlim=(-15, 230)):
    for Vs, (t, _, I) in fam.items():
        ax.plot(t, I[key], lw=1, label=f"{Vs} mV")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("t dall'inizio del gradino (ms)")
    ax.set_ylabel("I (nA)")
    ax.set_xlim(*xlim)
    ax.grid(alpha=0.3)


# Dizionario dei risultati: (valore del modello, valore di Wang, descrizione)
summary = {}

## VC1 – Corrente totale e blocco da TEA (Wang Fig. 2A-B)
Holding −80 mV, prepulse a −100 mV per 30 ms (rimuove l'inattivazione), poi gradini di 200 ms
da −80 a +60 mV ogni 20 mV. TEA 1 mM è simulato bloccando il `TEA_BLOCK` di I_Kv3.

In [ ]:
PRE = (-100.0, 30.0)
A_ctrl = step_family(-80, prepulse=PRE)
A_tea = step_family(-80, prepulse=PRE, block={"Kv3": TEA_BLOCK})
norm = value_at(A_ctrl, 60)          # corrente di controllo a +60 mV: riferimento di normalizzazione

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
plot_family(ax[0], A_ctrl, "VC1 controllo, hold −80 mV  [Wang 2A sx]")
plot_family(ax[1], A_tea, f"VC1 + TEA ({TEA_BLOCK:.0%} di I_Kv3)  [Wang 2A dx]")
ax[1].set_ylim(ax[0].get_ylim())
ax[0].legend(fontsize=7)
ax[2].plot(STEPS, iv(A_ctrl) / norm, "o-", mfc="w", color="k", label="controllo")
ax[2].plot(STEPS, iv(A_tea) / norm, "o-", color="k", label="TEA")
ax[2].set_title("I-V normalizzata a 195 ms  [Wang 2B]", fontsize=10)
ax[2].set_xlabel("V (mV)"); ax[2].set_ylabel("I / I_controllo(+60)"); ax[2].grid(alpha=0.3); ax[2].legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/vc1_fig2AB.png", dpi=120); plt.show()

summary["I tot a +60 (nA)"] = (norm, "> 15", "corrente totale massima, hold −80")
summary["TEA: I tot(+60) / controllo"] = (value_at(A_tea, 60) / norm, "0.361 ± 0.063", "Fig. 2B")
summary["TEA: I(−40) / controllo(−40)"] = (value_at(A_tea, -40) / value_at(A_ctrl, -40), "≈ 1", "traccia a −40 'praticamente identica'")

## VC2 – I_HT isolata tenendo il neurone a −40 mV (Wang Fig. 2C-D)
Wang tiene la cellula a −40 mV per 2 minuti: I_LT si inattiva e resta (quasi) solo I_HT.
Nell'inserto di Fig. 2C non c'è il prepulse a −100 mV (che rimuoverebbe di nuovo l'inattivazione),
quindi qui i gradini partono direttamente da −40 mV, con i gate allo stato stazionario a −40 mV.

In [ ]:
C_ctrl = step_family(-40)
C_tea = step_family(-40, block={"Kv3": TEA_BLOCK})

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
plot_family(ax[0], C_ctrl, "VC2 controllo, hold −40 mV  [Wang 2C sx]")
plot_family(ax[1], C_tea, "VC2 + TEA  [Wang 2C dx]")
ax[0].set_ylim(min(ax[0].get_ylim()[0], -0.5), ax[0].get_ylim()[1]); ax[1].set_ylim(ax[0].get_ylim())
ax[0].legend(fontsize=7)
ax[2].plot(STEPS, iv(C_ctrl) / norm, "o-", mfc="w", color="k", label="controllo (hold −40)")
ax[2].plot(STEPS, iv(C_tea) / norm, "o-", color="k", label="TEA (hold −40)")
ax[2].plot(STEPS, iv(C_ctrl, "LT") / norm, "g--", label="di cui I_LT residua (modello)")
ax[2].set_title("I-V normalizzata a I_controllo(+60) di VC1  [Wang 2D]", fontsize=10)
ax[2].set_xlabel("V (mV)"); ax[2].set_ylabel("I / I_controllo(+60, hold −80)"); ax[2].grid(alpha=0.3); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/vc2_fig2CD.png", dpi=120); plt.show()

summary["hold −40: I(+60) / controllo"] = (value_at(C_ctrl, 60) / norm, "0.772 ± 0.043", "frazione I_HT, Fig. 2D")
summary["hold −40 + TEA: I(+60) / controllo"] = (value_at(C_tea, 60) / norm, "0.202 ± 0.032", "Fig. 2D")
summary["hold −40: blocco TEA a +60"] = (1 - value_at(C_tea, 60) / value_at(C_ctrl, 60), "0.70–0.80", "testo")

## VC3 – I_LT per sottrazione (Wang Fig. 2E-F) e confronto con le correnti vere del modello
Wang ottiene I_LT come **VC1 − VC2** (totale da −80 meno I_HT da −40). Questa sottrazione funziona solo se
(1) tenere a −40 mV inattiva davvero I_LT e (2) non cambia I_HT. Nel modello conosciamo le correnti vere,
quindi possiamo controllare se l'ipotesi regge: confrontiamo la I_LT "sperimentale" (sottrazione) con
`I_LT` del modello nel protocollo VC1.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for Vs in STEPS:
    tA, _, I_A = A_ctrl[Vs]
    t, _, I_C = C_ctrl[Vs]
    # VC1 ha 30 ms di prepulse in più: si allineano le tracce sull'inizio del gradino (t = 0)
    ax[0].plot(t, np.interp(t, tA, I_A["total"]) - I_C["total"], lw=1, label=f"{Vs} mV")
ax[0].set_xlim(-5, 210); ax[0].set_title("I_LT per sottrazione VC1 − VC2  [Wang 2E]", fontsize=10)
ax[0].set_xlabel("t dall'inizio del gradino (ms)"); ax[0].set_ylabel("I (nA)"); ax[0].grid(alpha=0.3); ax[0].legend(fontsize=7)

lt_sub = iv(A_ctrl) - iv(C_ctrl)
ax[1].plot(STEPS, iv(C_ctrl) / norm, "o-", mfc="w", color="k", label="I_HT 'sperimentale' (= VC2)")
ax[1].plot(STEPS, lt_sub / norm, "o-", color="k", label="I_LT 'sperimentale' (VC1 − VC2)")
ax[1].plot(STEPS, iv(A_ctrl, "Kv3") / norm, "b--", label="I_Kv3 vera (VC1)")
ax[1].plot(STEPS, iv(A_ctrl, "LT") / norm, "g--", label="I_LT vera (VC1)")
ax[1].set_title("I_HT vs I_LT  [Wang 2F]", fontsize=10)
ax[1].set_xlabel("V (mV)"); ax[1].set_ylabel("I / I_controllo(+60)"); ax[1].grid(alpha=0.3); ax[1].legend(fontsize=8)

Vf = np.linspace(-110, 60, 300)
for g, c in zip(GATES, ["g", "olive", "b", "c"]):
    ax[2].plot(Vf, x_inf(g, Vf), color=c, label=f"{g}∞")
ax[2].axvline(-80, color="grey", ls=":"); ax[2].axvline(-40, color="grey", ls=":")
ax[2].set_title("Gate allo stato stazionario (linee: hold −80 e −40)", fontsize=10)
ax[2].set_xlabel("V (mV)"); ax[2].grid(alpha=0.3); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/vc3_fig2EF.png", dpi=120); plt.show()

print(f"r_inf(-80) = {x_inf('r', -80):.3f}, r_inf(-40) = {x_inf('r', -40):.3f}  "
      f"-> I_LT a hold -40 conserva il {x_inf('r', -40) / x_inf('r', -80):.0%} della disponibilità che ha a -80")


def threshold(I, steps=STEPS, frac=0.02):
    """Prima tensione del gradino a cui la corrente supera frac * il suo valore a +60 mV."""
    I = np.asarray(I)
    if np.abs(I).max() < 0.01 * abs(norm):     # corrente praticamente nulla: la soglia non ha senso
        return np.nan
    return steps[np.argmax(np.abs(I) > frac * np.abs(I[-1]))]


summary["I_LT / totale a +60 (sottrazione)"] = (lt_sub[-1] / norm, "0.229 ± 0.044", "Fig. 2E-F, metodo di Wang")
summary["I_LT / totale a +60 (vera)"] = (iv(A_ctrl, "LT")[-1] / norm, "0.229 ± 0.044", "corrente I_LT del modello")
summary["soglia I_HT (mV, 2% di I(+60))"] = (threshold(iv(C_ctrl)), "≈ −20", "testo")
summary["soglia I_LT (mV, 2% di I(+60), sottrazione)"] = (threshold(lt_sub), "≈ 20 mV sotto I_HT", "Fig. 2F")

## VC4 – I_LT isolata e blocco da DTX (Wang Fig. 2G-H)
Gradino da −80 a −20 mV (a −20 mV I_HT è ancora piccola) e poi a −40 mV, con e senza DTX 100 nM
(simulato bloccando il `DTX_BLOCK` di I_LT). La durata dei tratti è letta dalla figura (≈ 50 ms a −20).
In Fig. 2H: I-V con holding −80 mV fino a +20 mV, controllo vs DTX.

In [ ]:
protG = [(-80, 10), (-20, 50), (-40, 30)]
tG, VG, IG_ctrl, _ = clamp(protG)
_, _, IG_dtx, _ = clamp(protG, block={"LT": DTX_BLOCK})
iG = np.argmin(np.abs(tG - 55))      # fine del gradino a -20 mV

steps_H = np.arange(-80, 21, 10)
H_ctrl = step_family(-80, steps=steps_H)
H_dtx = step_family(-80, steps=steps_H, block={"LT": DTX_BLOCK})

fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].plot(tG - 10, IG_ctrl["total"] * 1000, "k", label="controllo")
ax[0].plot(tG - 10, IG_dtx["total"] * 1000, "r", label=f"DTX ({DTX_BLOCK:.0%} di I_LT)")
ax[0].set_title("−80 → −20 → −40 mV  [Wang 2G]", fontsize=10)
ax[0].set_xlabel("t (ms)"); ax[0].set_ylabel("I (pA)"); ax[0].grid(alpha=0.3); ax[0].legend()
ax[1].plot(tG - 10, IG_ctrl["LT"] * 1000, "g", label="I_LT (controllo)")
ax[1].plot(tG - 10, IG_ctrl["Kv3"] * 1000, "b", label="I_Kv3 (controllo)")
ax[1].set_title("Componenti del modello nel protocollo 2G", fontsize=10)
ax[1].set_xlabel("t (ms)"); ax[1].set_ylabel("I (pA)"); ax[1].grid(alpha=0.3); ax[1].legend()
ax[2].plot(steps_H, iv(H_ctrl, steps=steps_H), "o-", mfc="w", color="k", label="controllo")
ax[2].plot(steps_H, iv(H_dtx, steps=steps_H), "o-", color="k", label="DTX")
ax[2].set_title("I-V hold −80 mV ± DTX  [Wang 2H]", fontsize=10)
ax[2].set_xlabel("V (mV)"); ax[2].set_ylabel("I (nA)"); ax[2].grid(alpha=0.3); ax[2].legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/vc4_fig2GH.png", dpi=120); plt.show()

summary["I a −20 mV, hold −80 (pA)"] = (IG_ctrl["total"][iG] * 1000, "≈ 400 (letto da Fig. 2G)", "fine gradino")
summary["DTX: I(−20) / controllo"] = (IG_dtx["total"][iG] / IG_ctrl["total"][iG], "0.250 ± 0.021", "Fig. 2G")
summary["I-V a +20 mV, hold −80 (nA)"] = (value_at(H_ctrl, 20), "≈ 3 (letto da Fig. 2H)", "una sola cellula")

## VC5 – Cinetica di attivazione: tempo di salita 10–90% (Wang Fig. 4A-C)
Holding −40 mV (I_LT inattivata), gradini fino a +60 mV. Wang misura il tempo per passare dal 10% al 90%
dell'attivazione massima sulla corrente registrata. Qui lo misuriamo sulla corrente totale meno la
corrente di holding (come farebbe lo sperimentatore) e, per confronto, su I_Kv3 da sola.

In [ ]:
def rise_time(t, I, I_base=0.0):
    """Tempo 10-90% (ms) della corrente dopo il gradino (t >= 0), rispetto alla linea di base I_base."""
    sel = t >= 0
    tt, y = t[sel], I[sel] - I_base
    ymax = y.max()
    return tt[np.argmax(y >= 0.9 * ymax)] - tt[np.argmax(y >= 0.1 * ymax)]


Vr = [0, 20, 40, 60]
R = step_family(-40, steps=Vr, t_pre=5, t_step=50, t_post=10, block=None)
rt_tot, rt_kv3 = [], []
for Vs in Vr:
    t, _, I = R[Vs]
    base = I["total"][t < 0][-1]
    rt_tot.append(rise_time(t, I["total"], base))
    rt_kv3.append(rise_time(t, I["Kv3"], I["Kv3"][t < 0][-1]))
R_all = step_family(-40, t_pre=5, t_step=50, t_post=10)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
plot_family(ax[0], R_all, "Gradini da hold −40 mV  [Wang 4A]", xlim=(-2, 15))
ax[0].legend(fontsize=7)
ax[1].plot(Vr, rt_tot, "ko-", label="modello, corrente totale")
ax[1].plot(Vr, rt_kv3, "bo--", mfc="w", label="modello, solo I_Kv3")
ax[1].errorbar([0, 60], [5.53, 0.50], yerr=[0.65, 0.02], fmt="rs", capsize=4, label="Wang MNTB (n = 9)")
ax[1].set_title("Tempo di salita 10–90%  [Wang 4C]", fontsize=10)
ax[1].set_xlabel("V del gradino (mV)"); ax[1].set_ylabel("ms"); ax[1].grid(alpha=0.3); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/vc5_fig4AC.png", dpi=120); plt.show()

summary["rise time 10-90% a 0 mV (ms)"] = (rt_tot[0], "5.53 ± 0.65", "Fig. 4C")
summary["rise time 10-90% a +60 mV (ms)"] = (rt_tot[-1], "0.50 ± 0.02", "Fig. 4C")

## VC6 – Tail current a −20 mV e blocco da TEA (Wang Fig. 4D-G)
Holding −40 mV, breve gradino a +40 mV, poi ritorno a −20 mV: la corrente "di coda" decade mentre i canali
Kv3 si chiudono (deattivazione). Wang la fitta con un singolo esponenziale (τ = 5.2 ± 0.3 ms).
La durata del gradino a +40 mV non è riportata nel testo: `TAIL_PULSE` è letto dalla figura (approssimativo).
Il fit è fatto con un esponenziale più una costante, perché a −20 mV la corrente non torna a zero.

In [ ]:
TAIL_PULSE = 10.0   # ms a +40 mV (stima dalla Fig. 4D)
TAIL_DUR = 60.0     # ms a -20 mV
protT = [(-40, 5), (40, TAIL_PULSE), (-20, TAIL_DUR)]
tT, VT, IT_ctrl, _ = clamp(protT)
_, _, IT_tea, _ = clamp(protT, block={"Kv3": TEA_BLOCK})
t_tail0 = 5 + TAIL_PULSE
sel = tT >= t_tail0


def fit_tail(t, y):
    f = lambda t, A, tau, c: A * np.exp(-t / tau) + c
    popt, _ = curve_fit(f, t, y, p0=[y[0] - y[-1], 3.0, y[-1]])
    return popt


tail_t = tT[sel] - t_tail0
popt_tot = fit_tail(tail_t, IT_ctrl["total"][sel])
popt_kv3 = fit_tail(tail_t, IT_ctrl["Kv3"][sel])
i_tail_peak = np.argmax(sel)                       # primo campione a -20 mV
tea_inhib = 1 - IT_tea["total"][i_tail_peak] / IT_ctrl["total"][i_tail_peak]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].plot(tT - 5, IT_ctrl["total"], "k", label="controllo")
ax[0].plot(tT - 5, IT_tea["total"], "r", label="TEA")
ax[0].plot(tail_t + TAIL_PULSE, popt_tot[0] * np.exp(-tail_t / popt_tot[1]) + popt_tot[2], "y--",
           label=f"fit: τ = {popt_tot[1]:.2f} ms")
ax[0].set_title("hold −40 → +40 → −20 mV  [Wang 4D]", fontsize=10)
ax[0].set_xlabel("t (ms)"); ax[0].set_ylabel("I (nA)"); ax[0].grid(alpha=0.3); ax[0].legend()
taus = np.linspace(-60, 0, 100)
ax[1].plot(taus, tau_x("n", taus), "b", label="τ_n (modello)")
ax[1].plot(taus, tau_x("n", taus) / 3, "b:", label="τ_n / 3 (n³ che decade verso 0)")
ax[1].errorbar([-20], [5.2], yerr=[0.3], fmt="rs", capsize=4, label="τ tail Wang a −20 mV")
ax[1].set_title("Perché la tail del modello è veloce", fontsize=10)
ax[1].set_xlabel("V (mV)"); ax[1].set_ylabel("ms"); ax[1].grid(alpha=0.3); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/vc6_fig4DG.png", dpi=120); plt.show()

summary["τ tail a −20 mV, corrente totale (ms)"] = (popt_tot[1], "5.2 ± 0.3", "Fig. 4F")
summary["τ tail a −20 mV, solo I_Kv3 (ms)"] = (popt_kv3[1], "5.2 ± 0.3", "Fig. 4F")
summary["TEA: inibizione della tail"] = (tea_inhib, "0.750 ± 0.059", "Fig. 4G")

## Riepilogo: modello vs Wang 1998

In [ ]:
print(f"{'misura':<44}{'modello':>10}   {'Wang 1998':<26}note")
print("-" * 105)
for k, (mod, ref, note) in summary.items():
    print(f"{k:<44}{mod:>10.3f}   {ref:<26}{note}")

## Come leggere i risultati
- **VC1 (2A-B):** la corrente totale a +60 mV deve essere ≳ 15 nA e TEA deve lasciarne circa il 36%.
  Il residuo dipende sia da `TEA_BLOCK` sia da quanta I_LT c'è a +60 mV.
- **VC2 (2C-D):** se tenere a −40 mV inattivasse I_LT come nei neuroni veri, la curva "di cui I_LT residua"
  sarebbe vicina a zero. Nel modello il gate r dipende pochissimo dalla tensione (vedi pannello dei gate
  in VC3), quindi a −40 mV I_LT resta quasi tutta disponibile.
- **VC3 (2E-F):** la differenza tra I_LT "per sottrazione" e I_LT "vera" misura quanto il metodo di Wang
  applicato al modello sia affidabile. Se sono molto diverse, il confronto con Fig. 2F va fatto con la
  curva "per sottrazione", perché è quella che Wang ha misurato.
- **VC5 (4C):** un tempo di salita troppo lungo a +60 mV vuol dire che l'attivazione di Kv3 è troppo lenta
  ad alte tensioni (α_n cresce con eta_alpha_n = 0.04/mV).
- **VC6 (4D-G):** la corrente Kv3 va come n³: se n decade con τ_n, n³ decade più in fretta, con una τ
  apparente tra τ_n/3 e τ_n (dipende da quanto n resta aperto a −20 mV). Se anche τ_n è sotto i 5.2 ms di
  Wang, i canali Kv3 del modello si chiudono troppo in fretta qualunque sia il fit.
- Le misure "lette dalla figura" sono approssimative: usatele come ordine di grandezza.